# conv-output-shape — worked example 1: Predict conv2d output shape with asymmetric kernel/stride/padding

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-output-shape`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The 2-D convolution output-shape formula is `H_out = (H + 2*PH - KH) // SH + 1` and analogously for width. The batch axis `B` and the `out_channels` axis pass through unchanged; the input-channel axis `IC` is contracted away and never appears in the output shape. Because `//` is floor division, integer arithmetic alone predicts the exact shape produced by `nn.Conv2d`.

## Worked solution

We are given `input_shape=(2, 3, 28, 40)`, `out_channels=16`, `kernel_size=(5, 3)`, `stride=(2, 1)`, `padding=(1, 1)`.

**Step 1 — unpack the input shape.** `B=2, IC=3, H=28, W=40`. Note `IC=3` is only consumed by the kernel and will NOT appear in the result; it is replaced by `out_channels=16`.

**Step 2 — compute `H_out`.** Plug height values into the formula: `(28 + 2*1 - 5) // 2 + 1 = (28 + 2 - 5) // 2 + 1 = 25 // 2 + 1 = 12 + 1 = 13`. The floor division matters: `25 // 2 = 12`, not 12.5.

**Step 3 — compute `W_out`.** Width uses its own kernel/stride/padding: `(40 + 2*1 - 3) // 1 + 1 = (40 + 2 - 3) // 1 + 1 = 39 // 1 + 1 = 39 + 1 = 40`. With stride 1 and padding 1 and kernel 3, width is preserved (SAME-style).

**Step 4 — assemble.** Output shape = `(B, out_channels, H_out, W_out) = (2, 16, 13, 40)`.

**Why it works.** Convolution slides the kernel across each spatial axis independently, so height and width are computed with separate hyperparameters. The `// S + 1` counts how many valid kernel placements fit after padding. We verify against a real `nn.Conv2d` to confirm the analytic prediction.

In [ ]:
def conv2d_outshape(input_shape, out_channels, kernel_size, stride, padding):
    B, IC, H, W = input_shape
    KH, KW = kernel_size
    SH, SW = stride
    PH, PW = padding
    H_out = (H + 2 * PH - KH) // SH + 1
    W_out = (W + 2 * PW - KW) // SW + 1
    return (B, out_channels, H_out, W_out)


input_shape = (2, 3, 28, 40)
pred = conv2d_outshape(input_shape, 16, (5, 3), (2, 1), (1, 1))
conv = t.nn.Conv2d(3, 16, kernel_size=(5, 3), stride=(2, 1), padding=(1, 1))
actual = tuple(conv(t.zeros(input_shape)).shape)
print("predicted:", pred)
print("actual:   ", actual)
print("match:", pred == actual)